# Premier test RAG sans métadonnées

In [2]:
from src.backend.pgvector_store import get_pg_connection
from src.backend.vector_store import build_vector_store  # ton modèle e5
from sentence_transformers import SentenceTransformer
import numpy as np

/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1. Modèle pour encoder les questions lycéens
model = SentenceTransformer('intfloat/multilingual-e5-base')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8698.79it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# 2. Question lycéen
query = "BUT informatique apprentissage Bretagne pas cher"
query_emb = model.encode([query], normalize_embeddings=True)[0]

In [10]:
# 3. Recherche en base pgvector
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("""
    SELECT chunk_id, chunk_text, type_formation, commune, frais_scolarite,
           embedding <=> (%s::vector) AS distance
    FROM formations_chunks_vectors 
    ORDER BY embedding <=> (%s::vector)
    LIMIT 3;
""", (query_emb.tolist(), query_emb.tolist()))

In [11]:
results = cur.fetchall()
for row in results:
    print(f"Score: {row[5]:.3f} | {row[2]} à {row[3]} ({row[4]}€)")
    print(f"Texte: {row[1][:200]}...\n")

cur.close()
conn.close()

Score: 0.201 | BTS - BTSA - BTSM à Mably (None€)
Texte: Nom de la formation : BTS - Services - Métiers de la coiffure - en apprentissage

Type de formation : BTS - BTSA - BTSM

Établissement : CFA DU ROANNAIS ARPA (Mably - 42), Privés (Mably, Loire, Auverg...

Score: 0.203 | Formations des écoles d'ingénieurs à Chasseneuil-du-Poitou (7800€)
Texte: Nom de la formation : Formation d'ingénieur Bac + 5 - Voie d'accès réservée aux BAC +1 uniquement

Type de formation : Formations des écoles d'ingénieurs

Établissement : ESIGELEC Poitiers (Chasseneui...

Score: 0.204 | Licence,LPE - Licence Professorat des Ecoles à Strasbourg (178€)
Texte: Nom de la formation : Licence - Professorat des Ecoles - Lettres et arts

Type de formation : Licence,LPE - Licence Professorat des Ecoles

Établissement : Université de Strasbourg (67), Publics (Stra...

